<a href="https://colab.research.google.com/github/ingkevingarcia/Proyectos-ETL-y-Dashboard/blob/main/Pipelines/Covid-19/Proyecto_COVID_19_Con_Blob_Storage_De_Azure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalar SDK java 8

In [10]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Descargar Spark

In [11]:
!wget -q https://archive.apache.org/dist/spark/spark-3.3.4/spark-3.3.4-bin-hadoop3.tgz

# Descomprimir la version de Spark

In [12]:
!tar xf spark-3.3.4-bin-hadoop3.tgz

# Establecer las variables de entorno

In [13]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.4-bin-hadoop3"

# Descargar findspark

In [14]:
!pip install -q findspark

# Crear la sesión de Spark con las configuraciones necesarias para conectarse a AWS S3

In [15]:
import findspark

findspark.init()

from pyspark.sql import SparkSession

spark = (SparkSession
         .builder
         .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.11.469")
         .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
         .getOrCreate()
         )

# Extraer las credenciales del diccionario

In [16]:
from google.colab import userdata

accessKeyId=userdata.get('ACCESS_KEY')
secretAccessKey=userdata.get('SECRET_ACCESS_KEY')

# Establecer las configuraciones de Hodoop necesarias

In [17]:
sc = spark.sparkContext
sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', accessKeyId)
sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', secretAccessKey)
sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 's3.amazonaws.com')

#Proyecto COVID-19

1) A partir del archivo csv Case, determine las tres ciudades con más casos confirmados de la enfermedad. La salida debe contener tres columnas: provincia, ciudad y casos confirmados. El resultado debe contener exactamente los tres nombre de ciudades con más casos confirmados ya que no se admiten otros valores.



2) Cree un dataframe a partir del archivo csv PatientInfo. Asegúrese de que su dataframe no contenga pacientes duplicados.



2.1) ¿Cuántos pacientes tienen informado por quién se contagiaron(columna
infected_by)? Obtenga solo los pacientes que tengan informado por quién se contagiaron.



2.2) A partir de la salida del inciso anterior obtenga solo los pacientes femeninos. La salida no debe contener las columnas released_date y deceased_date.



2.3) Establezca el número de particiones del dataframe resultante del inciso anterior en dos. Escriba el dataframe resultante en un archivo parquet. La salida debe estar particionada por la provincia y el modo de escritura debe ser overwrite.



Punto 1

Leemos el archivo CSV de casos

In [18]:
Df_Casos = spark.read.option('header', 'true').option('inferSchema', 'true').csv('s3a://kevindatos/csv/Case.csv')

In [19]:
#Visualizacion del DF de casos
Df_Casos.show()

+--------+--------+---------------+-----+--------------------+---------+---------+----------+
| case_id|province|           city|group|      infection_case|confirmed| latitude| longitude|
+--------+--------+---------------+-----+--------------------+---------+---------+----------+
| 1000001|   Seoul|     Yongsan-gu| true|       Itaewon Clubs|      139|37.538621|126.992652|
| 1000002|   Seoul|      Gwanak-gu| true|             Richway|      119| 37.48208|126.901384|
| 1000003|   Seoul|        Guro-gu| true| Guro-gu Call Center|       95|37.508163|126.884387|
| 1000004|   Seoul|   Yangcheon-gu| true|Yangcheon Table T...|       43|37.546061|126.874209|
| 1000005|   Seoul|      Dobong-gu| true|     Day Care Center|       43|37.679422|127.044374|
| 1000006|   Seoul|        Guro-gu| true|Manmin Central Ch...|       41|37.481059|126.894343|
| 1000007|   Seoul|from other city| true|SMR Newly Planted...|       36|        -|         -|
| 1000008|   Seoul|  Dongdaemun-gu| true|       Dongan Churc

In [20]:
#Importamos funciones necesarias
from pyspark.sql.functions import desc
from pyspark.sql.functions import col

#Filtramos las ciudades que esten en nulas o no tengan nombre
Df_Filter = Df_Casos.filter((col('city') != '-') & (col('city') != 'from other city'))

#Sacamos el top 3 ciudades con mas contagios
Df_top3 =   Df_Filter.orderBy(col('confirmed').desc()).limit(3)

#Seleccionamos solo las columnas que nos competen y con sus respectivos alias establecidos
Df_punto1 = Df_top3.select(col('province').alias( 'Provincia'),
                           col('city').alias('Ciudad'),
                           col('confirmed').alias('Casos confirmados'))
#Mostramos los resultados
Df_punto1.show()

+---------+------------+-----------------+
|Provincia|      Ciudad|Casos confirmados|
+---------+------------+-----------------+
|    Daegu|      Nam-gu|             4511|
|    Daegu|Dalseong-gun|              196|
|    Seoul|  Yongsan-gu|              139|
+---------+------------+-----------------+



Punto 2

In [21]:
Df_Pacientes = spark.read.option('header', 'true').option('inferSchema', 'true').csv('s3a://kevindatos/csv/PatientInfo.csv')

In [22]:
#Visualizacion del DF de Pacientes
Df_Pacientes.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+-------------------+-------------------+-------------+--------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|     confirmed_date|      released_date|deceased_date|   state|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+-------------------+-------------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       null|            75|        2020-01-22|2020-01-23 00:00:00|2020-02-05 00:00:00|         null|released|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       null|            31|              null|2020-01-30 00:00:00|2020-03-02 00:00:00|         null|released|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 20020

In [23]:
#Miramos si hay pacientes duplicados
Df_Filter_Duplicados = Df_Pacientes.groupBy(col('patient_id')).count().filter(col("count") > 1).show()

#Eliminamos los pacientes duplicados
Df_Sin_Duplicados = Df_Pacientes.dropDuplicates(['patient_id'])

#Comprobamos que haya quedado bien
Df_Filter_Duplicados = Df_Sin_Duplicados.groupBy(col('patient_id')).count().filter(col("count") > 1).show()

+----------+-----+
|patient_id|count|
+----------+-----+
|1200012238|    2|
+----------+-----+

+----------+-----+
|patient_id|count|
+----------+-----+
+----------+-----+



Punto 2.1

In [24]:
#Se eliminan los pacientes que no saben por quien se contagiaron
Df_Filter_Contagiados = Df_Sin_Duplicados.filter(col('infected_by') .isNotNull())

#Cuantos pacientes saben por quien se contagiaron
Conteo = Df_Filter_Contagiados.count()
print('El numero de personas que conoce quienes lo contagiaron son: {} '.format(Conteo))

El numero de personas que conoce quienes lo contagiaron son: 1346 


Punto 2.2

In [25]:
#Solo dejamos los pacientes de genero femenino
Df_where = Df_Filter_Contagiados.where("Sex = 'female'")

#Eliminamos las columnas no deseadas
Df_punto2 = Df_where.drop('released_date','deceased_date')
Df_punto2.show()

+----------+------+---+-------+--------+-------------+--------------------+-----------+--------------+------------------+-------------------+--------+
|patient_id|   sex|age|country|province|         city|      infection_case|infected_by|contact_number|symptom_onset_date|     confirmed_date|   state|
+----------+------+---+-------+--------+-------------+--------------------+-----------+--------------+------------------+-------------------+--------+
|1000000005|female|20s|  Korea|   Seoul|  Seongbuk-gu|contact with patient| 1000000002|             2|              null|2020-01-31 00:00:00|released|
|1000000006|female|50s|  Korea|   Seoul|    Jongno-gu|contact with patient| 1000000003|            43|              null|2020-01-31 00:00:00|released|
|1000000010|female|60s|  Korea|   Seoul|  Seongbuk-gu|contact with patient| 1000000003|             6|              null|2020-02-05 00:00:00|released|
|1000000014|female|60s|  Korea|   Seoul|    Jongno-gu|contact with patient| 1000000013|       

Punto 2.3

In [26]:
#Particionamos la salida en 2 del punto 1
Df_Salida_1 = Df_punto1.coalesce(2)

#Escribimos el archivo del punto 12 en el bucket S3 de AWS
Df_Salida_1.write.mode('overwrite').parquet('s3a://kevindatos/salida/covid/punto_1')

#Escribimos el archivo del punto 2 en el bucket S3 de AWS particionando por provincia
Df_punto2.write.partitionBy('province').mode('overwrite').parquet('s3a://kevindatos/salida/covid/punto_2')